In [1]:
%pip install pymongo
import joblib
import re
import pandas as pd

from pymongo import MongoClient
from datetime import datetime, timedelta

model = joblib.load("../trained_models/logistic_regression_model.pkl")
vectorizer = joblib.load("../trained_models/tfidf_vectorizer.pkl")

print("Model and vectorizer loaded successfully.")


  Using cached pymongo-4.17.0-cp310-cp310-win_amd64.whl.metadata (10 kB)
  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
Using cached pymongo-4.17.0-cp310-cp310-win_amd64.whl (815 kB)
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Model and vectorizer loaded successfully.


In [2]:
def clean_log(text):
    text = str(text).lower()

    text = re.sub(r'\([^)]*\)', ' ', text)
    text = re.sub(r'0x[a-fA-F0-9]+', ' ', text)
    text = re.sub(r'[/\\][^\s]+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def predict_log(log_text):
    clean_text = clean_log(log_text)

    vector = vectorizer.transform([clean_text])

    prediction = model.predict(vector)[0]
    probabilities = model.predict_proba(vector)[0]

    confidence = float(round(probabilities.max() * 100, 2))

    return {
        "original_log": log_text,
        "clean_log": clean_text,
        "predicted_incident": prediction,
        "confidence": confidence
    }

In [3]:
def get_top_keywords(log_text, top_n=5):
    clean_text = clean_log(log_text)
    vector = vectorizer.transform([clean_text])

    prediction = model.predict(vector)[0]
    class_index = list(model.classes_).index(prediction)

    feature_names = vectorizer.get_feature_names_out()

    tfidf_scores = vector.toarray()[0]
    class_weights = model.coef_[class_index]

    contribution_scores = tfidf_scores * class_weights

    top_indices = contribution_scores.argsort()[-top_n:][::-1]

    keywords = [
        feature_names[i]
        for i in top_indices
        if contribution_scores[i] > 0
    ]

    return keywords

In [4]:
client = MongoClient("mongodb://localhost:27017/")

db = client["AI_Log_Analyzer"]
collection = db["log_history"]

print("MongoDB connected successfully.")

MongoDB connected successfully.


In [5]:
sample_logs = [
    {
        "timestamp": datetime.now() - timedelta(hours=3),
        "node": "R30-M0-N9-C:J16-U01",
        "raw_log": "instruction cache parity error corrected"
    },
    {
        "timestamp": datetime.now() - timedelta(hours=2),
        "node": "R30-M0-N9-C:J16-U01",
        "raw_log": "instruction cache parity error corrected"
    },
    {
        "timestamp": datetime.now() - timedelta(hours=1),
        "node": "R30-M0-N9-C:J16-U01",
        "raw_log": "machine check interrupt detected"
    },
    {
        "timestamp": datetime.now() - timedelta(minutes=45),
        "node": "R12-M1-N4-I:J18-U11",
        "raw_log": "file access denied during mount operation"
    },
    {
        "timestamp": datetime.now() - timedelta(minutes=30),
        "node": "R12-M1-N4-I:J18-U11",
        "raw_log": "lustre mount failed no such file or directory"
    },
    {
        "timestamp": datetime.now() - timedelta(minutes=10),
        "node": "R30-M0-N9-C:J16-U01",
        "raw_log": "instruction cache parity error corrected"
    }
]

In [6]:
processed_logs = []

for item in sample_logs:
    prediction = predict_log(item["raw_log"])
    keywords = get_top_keywords(item["raw_log"])

    processed_logs.append({
        "timestamp": item["timestamp"],
        "node": item["node"],
        "raw_log": item["raw_log"],
        "clean_log": prediction["clean_log"],
        "incident_type": prediction["predicted_incident"],
        "confidence": prediction["confidence"],
        "keywords": keywords
    })

collection.insert_many(processed_logs)

print("Logs inserted successfully.")

Logs inserted successfully.


In [7]:
logs_from_db = list(collection.find({}, {"_id": 0}))

df_logs = pd.DataFrame(logs_from_db)

df_logs.head()

,timestamp,node,raw_log,clean_log,incident_type,confidence,keywords
0,2026-07-07 10:30:08.361,R30-M0-N9-C:J16-U01,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error,100.00,"[instruction, corrected, cache, parity, parity..."
1,2026-07-07 11:30:08.361,R30-M0-N9-C:J16-U01,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error,100.00,"[instruction, corrected, cache, parity, parity..."
2,2026-07-07 12:30:08.361,R30-M0-N9-C:J16-U01,machine check interrupt detected,machine check interrupt detected,System Interrupt,99.98,"[interrupt, machine check, machine, check, che..."
3,2026-07-07 12:45:08.361,R12-M1-N4-I:J18-U11,file access denied during mount operation,file access denied during mount operation,Storage/File System,72.44,"[mount, file, denied]"
4,2026-07-07 13:00:08.361,R12-M1-N4-I:J18-U11,lustre mount failed no such file or directory,lustre mount failed no such file or directory,Storage/File System,100.00,"[mount, failed, directory, mount failed, lustr..."


In [8]:
df_logs["timestamp"] = pd.to_datetime(df_logs["timestamp"])

now = datetime.now()
last_1h = df_logs[df_logs["timestamp"] >= now - timedelta(hours=1)]
previous_1h = df_logs[
    (df_logs["timestamp"] < now - timedelta(hours=1)) &
    (df_logs["timestamp"] >= now - timedelta(hours=2))
]

current_counts = last_1h["incident_type"].value_counts()
previous_counts = previous_1h["incident_type"].value_counts()

insights = []

for incident in current_counts.index:
    current = current_counts.get(incident, 0)
    previous = previous_counts.get(incident, 0)

    if previous == 0 and current > 0:
        insights.append(f"⚠️ {incident} appeared {current} times in the last 1 hour, but was not seen in the previous hour.")
    else:
        change = ((current - previous) / previous) * 100
        if change >= 50:
            insights.append(f"🚨 {incident} increased by {change:.1f}% in the last 1 hour.")

insights

['⚠️ Storage/File System appeared 2 times in the last 1 hour, but was not seen in the previous hour.',
 '⚠️ Memory Error appeared 1 times in the last 1 hour, but was not seen in the previous hour.']

In [9]:
node_counts = last_1h["node"].value_counts()

node_insights = []

for node, count in node_counts.items():
    if count >= 2:
        node_insights.append(
            f"🚨 Node {node} generated {count} incidents in the last 1 hour. This may indicate localized instability."
        )

node_insights

['🚨 Node R12-M1-N4-I:J18-U11 generated 2 incidents in the last 1 hour. This may indicate localized instability.']

In [10]:
all_insights = insights + node_insights

for item in all_insights:
    print(item)

⚠️ Storage/File System appeared 2 times in the last 1 hour, but was not seen in the previous hour.
⚠️ Memory Error appeared 1 times in the last 1 hour, but was not seen in the previous hour.
🚨 Node R12-M1-N4-I:J18-U11 generated 2 incidents in the last 1 hour. This may indicate localized instability.


In [11]:
def calculate_risk_level(insights):
    critical_count = sum(1 for item in insights if "🚨" in item)
    warning_count = sum(1 for item in insights if "⚠️" in item)

    if critical_count >= 2:
        return "CRITICAL"
    elif critical_count == 1 or warning_count >= 2:
        return "HIGH"
    elif warning_count == 1:
        return "MEDIUM"
    else:
        return "LOW"

risk_level = calculate_risk_level(all_insights)

print("Risk Level:", risk_level)

Risk Level: HIGH


In [12]:
def calculate_risk_level(insights):
    critical_count = sum(1 for item in insights if "🚨" in item)
    warning_count = sum(1 for item in insights if "⚠️" in item)

    if critical_count >= 2:
        return "CRITICAL"
    elif critical_count == 1 or warning_count >= 2:
        return "HIGH"
    elif warning_count == 1:
        return "MEDIUM"
    else:
        return "LOW"

risk_level = calculate_risk_level(all_insights)

print("Risk Level:", risk_level)

Risk Level: HIGH
